# 06 — Train the home-win model with MLflow

Train three competing classifiers that each output a **probability the home team wins**, and
log everything to MLflow: parameters, metrics, an **ROC curve**, and a **calibration curve**
(critical — we care that "70%" really means a 70% win rate, not just ranking).

| Model | Why it's here |
|-------|---------------|
| Logistic Regression (scaled) | Transparent, well-calibrated baseline |
| Random Forest | Non-linear interactions, little tuning |
| XGBoost | Usually the strongest tabular model |

**No leakage in the split:** the most recent season (`NBA_HOLDOUT_SEASON`) is held out
entirely for the honest evaluation in notebook `07`. We train on the earlier seasons only.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (log_loss, brier_score_loss, roc_auc_score, accuracy_score,
                             roc_curve)
from sklearn.calibration import calibration_curve

import mlflow
from mlflow.models import infer_signature

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG  = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA   = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
GOLD_SCHEMA = f"{UC_SCHEMA}_gold"
FEATURES    = f"{UC_CATALOG}.{GOLD_SCHEMA}.game_features"
EXPERIMENT  = os.getenv("MLFLOW_EXPERIMENT_NAME", "/Users/alexander.booth@databricks.com/basketball-reference-waf")
HOLDOUT     = int(os.getenv("NBA_HOLDOUT_SEASON", "2025"))

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT)

FEATURE_COLS = [
    "win_pct_diff", "margin_diff", "net_rtg_diff",
    "efg_for_diff", "tov_for_diff", "orb_for_diff", "ftr_for_diff",
    "efg_against_diff", "tov_against_diff", "drb_against_diff", "ftr_against_diff",
    "rest_diff",
]
print("Experiment :", EXPERIMENT)
print("Holdout    :", HOLDOUT, "(reserved for notebook 07)")
print("Features   :", len(FEATURE_COLS))

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Experiment : /Users/alexander.booth@databricks.com/basketball-reference-waf
Holdout    : 2025 (reserved for notebook 07)
Features   : 12


## Load features and make the train / validation split

In [2]:
pdf = spark.table(FEATURES).toPandas()
train_pool = pdf[pdf["season"] != HOLDOUT].copy()
print(f"Total games: {len(pdf)} | training pool (excl. holdout): {len(train_pool)}")

X = train_pool[FEATURE_COLS].astype(float)
y = train_pool["home_win"].astype(int)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train={len(X_train)}  val={len(X_val)}  | home-win base rate (train): {y_train.mean():.3f}")

Total games: 3222 | training pool (excl. holdout): 2147
train=1717  val=430  | home-win base rate (train): 0.557


## Helper — wrap any estimator as a P(home win) pyfunc, evaluate, and log to MLflow

In [3]:
class HomeWinModel(mlflow.pyfunc.PythonModel):
    """Returns the probability that the HOME team wins."""
    def __init__(self, estimator):
        self.estimator = estimator
    def predict(self, context, model_input):
        return self.estimator.predict_proba(model_input)[:, 1]


def train_and_log(estimator, run_name, family, params):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("demo", "home_win")
        mlflow.set_tag("model_family", family)
        mlflow.log_params(params)

        estimator.fit(X_train, y_train)
        proba = estimator.predict_proba(X_val)[:, 1]
        metrics = {
            "val_log_loss": log_loss(y_val, proba),
            "val_brier":    brier_score_loss(y_val, proba),
            "val_roc_auc":  roc_auc_score(y_val, proba),
            "val_accuracy": accuracy_score(y_val, (proba >= 0.5).astype(int)),
        }
        mlflow.log_metrics(metrics)

        # ROC curve
        fpr, tpr, _ = roc_curve(y_val, proba)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr, label=f"AUC={metrics['val_roc_auc']:.3f}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.set_title(f"ROC - {run_name}"); ax.legend(); plt.tight_layout()
        mlflow.log_figure(fig, "roc_curve.png"); plt.close(fig)

        # Calibration curve
        frac_pos, mean_pred = calibration_curve(y_val, proba, n_bins=10, strategy="quantile")
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="perfect")
        ax.plot(mean_pred, frac_pos, marker="o", label=run_name)
        ax.set_xlabel("Predicted P(home win)"); ax.set_ylabel("Actual home-win rate")
        ax.set_title(f"Calibration - {run_name}"); ax.legend(); plt.tight_layout()
        mlflow.log_figure(fig, "calibration_curve.png"); plt.close(fig)

        sig = infer_signature(X_train, estimator.predict_proba(X_train.head(5))[:, 1])
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=HomeWinModel(estimator),
            signature=sig,
            input_example=X_train.head(3),
            pip_requirements=["scikit-learn", "xgboost", "pandas", "numpy"],
        )
        print(f"  {run_name:<16} "
              f"logloss={metrics['val_log_loss']:.4f}  auc={metrics['val_roc_auc']:.4f}  "
              f"acc={metrics['val_accuracy']:.4f}  run_id={run.info.run_id}")
        return {"run_name": run_name, "family": family, **metrics}

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## Train the three models

In [4]:
results = []
results.append(train_and_log(
    Pipeline([("scaler", StandardScaler()),
              ("lr", LogisticRegression(max_iter=1000, C=1.0))]),
    run_name="logreg_v1", family="logreg",
    params={"model": "logistic_regression", "C": 1.0, "scaled": True},
))
results.append(train_and_log(
    RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=30,
                           random_state=42, n_jobs=-1),
    run_name="random_forest_v1", family="random_forest",
    params={"model": "random_forest", "n_estimators": 300, "max_depth": 6, "min_samples_leaf": 30},
))
results.append(train_and_log(
    XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                  subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                  random_state=42, n_jobs=-1),
    run_name="xgboost_v1", family="xgboost",
    params={"model": "xgboost", "n_estimators": 300, "max_depth": 3, "learning_rate": 0.05},
))

print("\nValidation leaderboard (lower log-loss is better):")
import pandas as pd
print(pd.DataFrame(results).sort_values("val_log_loss").to_string(index=False))

2026/06/02 13:57:54 INFO mlflow.pyfunc: Validating input example against model signature


/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.05it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.05it/s]

...

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  7.22it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  8.94it/s]

  xgboost_v1       logloss=0.6686  auc=0.6520  acc=0.6209  run_id=25b04d684ba04f23a92ea8efa13587e7


🏃 View run xgboost_v1 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170/runs/25b04d684ba04f23a92ea8efa13587e7
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170



Validation leaderboard (lower log-loss is better):
        run_name        family  val_log_loss  val_brier  val_roc_auc  val_accuracy
       logreg_v1        logreg      0.632306   0.221744     0.686718      0.632558
random_forest_v1 random_forest      0.642003   0.226046     0.674801      0.618605
      xgboost_v1       xgboost      0.668610   0.235421     0.651953      0.620930


Three runs are logged with metrics + ROC + calibration curves. **Next:**
`07_evaluate_and_register` scores them on the held-out season and registers the champion.